# WTI COT MM Nowcasting — Kalman Filter — 05 Final Model

Fits the **winning KF model** (identified in notebook 04) on the full sample and produces:

1. Full-history filter and smoother estimates with uncertainty bands  
2. Prediction interval for the **next COT report** (live nowcast)  
3. Model diagnostics on one-step-ahead residuals  
4. Exported artifacts for the dashboard  

> **Note:** Even if a linear model ranked highest on OOS Spearman ρ in notebook 04, we fit the best *KF* variant here — the KF adds value through its uncertainty quantification and state tracking that a static regression cannot provide.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import sys
sys.path.append('../../../')

In [ ]:
import json
import pathlib
import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats

from statsmodels.tsa.statespace.mlemodel import MLEModel
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.stats.stattools import jarque_bera

plt.rcParams['figure.figsize'] = (14, 4)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

OUT_DIR = pathlib.Path('../../../cache/output/wti/mm')

In [ ]:
from src.utils.io.read import PreprocessedDataReader
from src.preprocessing.base import FutureTicker
from src.settings import Settings

pdr = PreprocessedDataReader(Settings.historical.paths.PREPROCESSED_DATA_PATH)
dataset = pdr.read_dataset(ticker=FutureTicker.WTI)
dataset['tradeDate'] = pd.to_datetime(dataset['tradeDate'])
dataset.sort_values('tradeDate', inplace=True)
dataset.reset_index(drop=True, inplace=True)

with open(OUT_DIR / 'kf_config.json') as f:
    kf_config = json.load(f)

with open(OUT_DIR / 'kf_comparison_results.json') as f:
    comparison = json.load(f)

RESPONSES   = kf_config['responses_raw']
FEATURES    = kf_config['selected_features']
RESP_LABELS = ['Net', 'Long', 'Short']
BEST_MODEL  = comparison['best_model']

print(f'Best KF model from notebook 04: {BEST_MODEL}')

---
## 1. Data Preparation

In [ ]:
# Use the joint-clean dataset (responses + features both non-NaN)
cols_needed = ['tradeDate'] + RESPONSES + FEATURES
df = dataset[cols_needed].dropna().reset_index(drop=True)

dates   = df['tradeDate'].values
Y_raw   = df[RESPONSES].values.astype(float)   # (T, 3)
X_raw   = df[FEATURES].values.astype(float)    # (T, 5)

Y_mean, Y_std = Y_raw.mean(axis=0), Y_raw.std(axis=0)
X_mean, X_std = X_raw.mean(axis=0), X_raw.std(axis=0)
Y = (Y_raw - Y_mean) / Y_std
X = (X_raw - X_mean) / X_std

k_endog    = Y.shape[1]   # 3
n_features = X.shape[1]   # 5

print(f'Full sample: {len(df)} observations')
print(f'Latest date: {df["tradeDate"].max().date()}')

---
## 2. Model Classes

Redefined here for notebook self-containment (no cross-notebook imports).

In [ ]:
class MultivariateLocalLevel(MLEModel):
    """Local Level — diagonal or full covariance."""
    def __init__(self, endog, full_cov=False):
        k = endog.shape[1]
        self.k_ = k; self.full_cov = full_cov
        super().__init__(endog, k_states=k, k_posdef=k)
        self['design'] = np.eye(k)
        self['transition'] = np.eye(k)
        self['selection'] = np.eye(k)
        self.initialize_approximate_diffuse()
    @property
    def param_names(self):
        k = self.k_
        n = [f'log_h_{i}' for i in range(k)] + [f'log_q_{i}' for i in range(k)]
        if self.full_cov:
            n += [f'h_cov_{i}{j}' for i in range(1, k) for j in range(i)]
        return n
    @property
    def start_params(self):
        s = np.log(np.std(self.endog, axis=0) + 1e-6)
        p = np.concatenate([s, s - 1.0])
        if self.full_cov:
            p = np.concatenate([p, np.zeros(self.k_ * (self.k_ - 1) // 2)])
        return p
    def update(self, params, **kwargs):
        params = super().update(params, **kwargs)
        k = self.k_
        h_s = np.exp(params[:k]); q_s = np.exp(params[k:2*k])
        if self.full_cov:
            L = np.diag(h_s)
            for idx_off, (i, j) in enumerate([(i,j) for i in range(1,k) for j in range(i)]):
                L[i, j] = params[2*k + idx_off]
            H = L @ L.T
        else:
            H = np.diag(h_s ** 2)
        self['obs_cov'] = H
        self['state_cov'] = np.diag(q_s ** 2)


class MultivariateLocalLinearTrend(MLEModel):
    """Local Linear Trend — diagonal covariances."""
    def __init__(self, endog):
        k = endog.shape[1]; self.k_ = k
        super().__init__(endog, k_states=2*k, k_posdef=2*k)
        Z = np.zeros((k, 2*k)); Z[:, :k] = np.eye(k)
        self['design'] = Z
        T = np.eye(2*k); T[:k, k:] = np.eye(k)
        self['transition'] = T
        self['selection'] = np.eye(2*k)
        self.initialize_approximate_diffuse()
    @property
    def param_names(self):
        k = self.k_
        return ([f'log_h_{i}' for i in range(k)] +
                [f'log_q1_{i}' for i in range(k)] +
                [f'log_q2_{i}' for i in range(k)])
    @property
    def start_params(self):
        s = np.log(np.std(self.endog, axis=0) + 1e-6)
        return np.concatenate([s, s - 1.0, s - 3.0])
    def update(self, params, **kwargs):
        params = super().update(params, **kwargs)
        k = self.k_
        h_s = np.exp(params[:k]); q1_s = np.exp(params[k:2*k]); q2_s = np.exp(params[2*k:])
        self['obs_cov'] = np.diag(h_s ** 2)
        Q = np.zeros((2*k, 2*k))
        Q[:k, :k] = np.diag(q1_s ** 2); Q[k:, k:] = np.diag(q2_s ** 2)
        self['state_cov'] = Q


class LocalLevelWithFeatures(MLEModel):
    """Local Level + fixed exogenous regression in observation equation."""
    def __init__(self, endog, exog):
        k = endog.shape[1]; p = exog.shape[1]
        self.k_ = k; self.p_ = p
        super().__init__(endog, k_states=k, k_posdef=k)
        self['design'] = np.eye(k)
        self['transition'] = np.eye(k)
        self['selection'] = np.eye(k)
        self._exog = np.array(exog)
        self.initialize_approximate_diffuse()
    @property
    def param_names(self):
        k, p = self.k_, self.p_
        return ([f'log_h_{i}' for i in range(k)] +
                [f'log_q_{i}' for i in range(k)] +
                [f'beta_{i}_{j}' for i in range(k) for j in range(p)])
    @property
    def start_params(self):
        k, p = self.k_, self.p_
        s = np.log(np.std(self.endog, axis=0) + 1e-6)
        return np.concatenate([s, s - 1.0, np.zeros(k * p)])
    def update(self, params, **kwargs):
        params = super().update(params, **kwargs)
        k, p = self.k_, self.p_
        h_s = np.exp(params[:k]); q_s = np.exp(params[k:2*k])
        beta = params[2*k:].reshape(k, p)
        self['obs_cov'] = np.diag(h_s ** 2)
        self['state_cov'] = np.diag(q_s ** 2)
        self['obs_intercept'] = beta @ self._exog.T


class DynamicRegression(MLEModel):
    """Dynamic regression — time-varying β_t as state."""
    def __init__(self, endog, exog):
        k = endog.shape[1]; p = exog.shape[1]; T = endog.shape[0]
        self.k_ = k; self.p_ = p
        super().__init__(endog, k_states=k*p, k_posdef=k*p)
        Z = np.zeros((k, k*p, T))
        for t in range(T):
            for i in range(k):
                Z[i, i*p:(i+1)*p, t] = exog[t]
        self['design'] = Z
        self['transition'] = np.eye(k*p)
        self['selection'] = np.eye(k*p)
        self.initialize_approximate_diffuse()
    @property
    def param_names(self):
        k, p = self.k_, self.p_
        return ([f'log_h_{i}' for i in range(k)] +
                [f'log_q_{i}_{j}' for i in range(k) for j in range(p)])
    @property
    def start_params(self):
        k, p = self.k_, self.p_
        s = np.log(np.std(self.endog, axis=0) + 1e-6)
        return np.concatenate([s, np.full(k * p, -2.0)])
    def update(self, params, **kwargs):
        params = super().update(params, **kwargs)
        k, p = self.k_, self.p_
        h_s = np.exp(params[:k]); q_s = np.exp(params[k:])
        self['obs_cov'] = np.diag(h_s ** 2)
        self['state_cov'] = np.diag(q_s ** 2)


# Map model name → constructor
def build_model(model_name, Y, X):
    needs_features = model_name in ('D: LL+Feat', 'E: Dyn-Reg')
    if model_name in ('A: LL-diag',):
        return MultivariateLocalLevel(Y, full_cov=False)
    elif model_name == 'B: LL-full':
        return MultivariateLocalLevel(Y, full_cov=True)
    elif model_name == 'C: LLT':
        return MultivariateLocalLinearTrend(Y)
    elif model_name == 'D: LL+Feat':
        return LocalLevelWithFeatures(Y, X)
    elif model_name == 'E: Dyn-Reg':
        return DynamicRegression(Y, X)
    else:
        raise ValueError(f'Unknown model: {model_name}')

print('Model classes defined.')

---
## 3. Full-Sample Fit

In [ ]:
print(f'Fitting: {BEST_MODEL} on full sample ({len(Y)} observations)...')

mod = build_model(BEST_MODEL, Y, X)
res = mod.fit(disp=False, method='lbfgs', maxiter=600)

print(f'Converged: {res.mle_retvals["converged"] if hasattr(res, "mle_retvals") else "N/A"}')
print(f'Log-likelihood : {res.llf:.4f}')
print(f'AIC            : {res.aic:.4f}')
print(f'BIC            : {res.bic:.4f}')
print(f'n params       : {len(res.params)}')

In [ ]:
# Print fitted parameters with standard errors
res.summary()

In [ ]:
# Estimated noise covariances (back-transform from log-params)
k = k_endog
H_diag = np.exp(res.params[:k]) ** 2
Q_diag = np.exp(res.params[k:2*k]) ** 2
SNR    = Q_diag / (H_diag + 1e-10)

noise_df = pd.DataFrame({
    'Response':   RESP_LABELS,
    'Obs noise σ (H)':   np.sqrt(H_diag).round(4),
    'State noise σ (Q)': np.sqrt(Q_diag).round(4),
    'SNR (Q/H)':         SNR.round(4),
})
noise_df.set_index('Response')

---
## 4. Filter and Smoother — Full History

- **Filter** $E[\alpha_t \mid y_{1:t}]$: what the model knew in real time  
- **Smoother** $E[\alpha_t \mid y_{1:T}]$: retrospective best estimate  
- **Prediction intervals**: derived from the predictive covariance $Z P_{t|t-1} Z^\top + H$

In [ ]:
# Extract filter and smoother (first k states = level for all models)
filtered_states  = res.filtered_state[:k].T    # (T, 3) scaled
smoothed_states  = res.smoothed_state[:k].T    # (T, 3) scaled

# Filtered state covariance → prediction intervals
filtered_cov = res.filtered_state_cov          # (k_states, k_states, T)
smoothed_cov = res.smoothed_state_cov          # (k_states, k_states, T)

# For Local Level: Var[y_{t+1}|y_{1:t}] = Var[alpha_{t+1|t}] + H
# Var[alpha_{t+1|t}] ≈ P_{t|t} + Q  (one-step predicted covariance)
H_mat = mod['obs_cov']   # (k, k)
Q_mat = mod['state_cov'][:k, :k]   # (k, k)

pred_var = np.array([
    np.diag(filtered_cov[:k, :k, t] + Q_mat + H_mat)
    for t in range(len(dates))
])  # (T, 3)

pred_std_orig = np.sqrt(np.maximum(pred_var, 0)) * Y_std   # back-transform

# Back-transform states to original scale
filtered_orig = filtered_states * Y_std + Y_mean
smoothed_orig = smoothed_states * Y_std + Y_mean
smoothed_std_orig = np.sqrt(np.array([
    np.diag(smoothed_cov[:k, :k, t])
    for t in range(len(dates))
])) * Y_std   # (T, 3)

print('Filter/smoother extracted.')

In [ ]:
# Full-history plot: observed, filtered, smoothed + 95% PI
Z_95 = 1.96
fig, axes = plt.subplots(k, 1, figsize=(16, 4 * k), sharex=True)

for i, (ax, label) in enumerate(zip(axes, RESP_LABELS)):
    ax.bar(dates, Y_raw[:, i], color='lightsteelblue', width=5,
           alpha=0.5, label='Observed Δ')

    # Smoother with CI
    ax.plot(dates, smoothed_orig[:, i], color='navy', linewidth=1.4,
            linestyle='--', label='Smoothed state')
    ax.fill_between(
        dates,
        smoothed_orig[:, i] - Z_95 * smoothed_std_orig[:, i],
        smoothed_orig[:, i] + Z_95 * smoothed_std_orig[:, i],
        alpha=0.15, color='navy', label='Smoother 95% CI',
    )

    # Filter
    ax.plot(dates, filtered_orig[:, i], color='darkorange', linewidth=1.1,
            label='Filtered state')

    ax.axhline(0, color='grey', linestyle=':', linewidth=0.8)
    ax.set_title(f'{label} Position Change', fontweight='bold')
    ax.set_ylabel('Contracts')
    ax.legend(loc='upper right', fontsize=8, ncol=2)

axes[-1].set_xlabel('Date')
plt.suptitle(f'{BEST_MODEL} — Full History: Filter, Smoother & 95% CI', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Zoom: last 3 years
N_RECENT = 156   # 3 years
fig, axes = plt.subplots(k, 1, figsize=(16, 4 * k), sharex=True)

for i, (ax, label) in enumerate(zip(axes, RESP_LABELS)):
    d  = dates[-N_RECENT:]
    obs = Y_raw[-N_RECENT:, i]
    sm  = smoothed_orig[-N_RECENT:, i]
    sm_s = smoothed_std_orig[-N_RECENT:, i]
    fl  = filtered_orig[-N_RECENT:, i]
    pi_s = pred_std_orig[-N_RECENT:, i]

    ax.bar(d, obs, color='lightsteelblue', width=5, alpha=0.5, label='Observed Δ')
    ax.plot(d, sm, color='navy',      linewidth=1.4, linestyle='--', label='Smoothed')
    ax.plot(d, fl, color='darkorange', linewidth=1.1, label='Filtered')
    ax.fill_between(d, fl - Z_95 * pi_s, fl + Z_95 * pi_s,
                    alpha=0.2, color='darkorange', label='Filter 95% PI')
    ax.fill_between(d, sm - Z_95 * sm_s, sm + Z_95 * sm_s,
                    alpha=0.15, color='navy')
    ax.axhline(0, color='grey', linestyle=':', linewidth=0.8)
    ax.set_title(f'{label} — Last 3 Years', fontweight='bold')
    ax.set_ylabel('Contracts')
    ax.legend(loc='upper right', fontsize=8, ncol=2)

axes[-1].set_xlabel('Date')
plt.suptitle(f'{BEST_MODEL} — Recent History Detail', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

---
## 5. One-Step-Ahead Residual Diagnostics

If the model is correctly specified, the **innovation sequence** (one-step prediction errors) should be:
- Zero-mean  
- Serially uncorrelated  
- Approximately Gaussian

In [ ]:
# Innovations: y_t - E[y_t | y_{1:t-1}]  (in original scale)
forecast_errors = res.forecasts_error[:k].T * Y_std   # (T, 3)

fig, axes = plt.subplots(k, 3, figsize=(16, 3.5 * k))

for i, (label, color) in enumerate(zip(RESP_LABELS, ['steelblue', 'green', 'tomato'])):
    err = pd.Series(forecast_errors[:, i]).dropna()

    # Time series
    axes[i, 0].plot(dates[:len(err)], err.values, color=color, linewidth=0.8)
    axes[i, 0].axhline(0, color='grey', linestyle='--')
    axes[i, 0].set_title(f'{label} — Innovations')
    axes[i, 0].set_ylabel('Contracts')

    # Distribution
    err.hist(bins=40, ax=axes[i, 1], color=color, alpha=0.7, density=True)
    xg = np.linspace(err.min(), err.max(), 200)
    axes[i, 1].plot(xg, stats.norm.pdf(xg, err.mean(), err.std()),
                    'k--', linewidth=1.2, label='Normal')
    axes[i, 1].set_title('Distribution')
    axes[i, 1].legend(fontsize=8)

    # ACF
    from statsmodels.graphics.tsaplots import plot_acf
    plot_acf(err, lags=20, ax=axes[i, 2], alpha=0.05, color=color)
    axes[i, 2].set_title('ACF of Innovations')

plt.suptitle(f'{BEST_MODEL} — Innovation Diagnostics', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Ljung-Box and Jarque-Bera tests
diag_rows = []
for i, label in enumerate(RESP_LABELS):
    err = pd.Series(forecast_errors[:, i]).dropna()
    lb  = acorr_ljungbox(err, lags=[8], return_df=True).iloc[0]
    jb_stat, jb_p, _, _ = jarque_bera(err)
    diag_rows.append({
        'Response':    label,
        'Mean':        round(err.mean(), 2),
        'Std':         round(err.std(), 2),
        'LB(8) stat':  round(lb['lb_stat'], 3),
        'LB(8) p-val': round(lb['lb_pvalue'], 4),
        'JB stat':     round(jb_stat, 3),
        'JB p-val':    round(jb_p, 4),
        'Serially uncorrelated?': 'Yes' if lb['lb_pvalue'] > 0.05 else 'No',
        'Approximately normal?':  'Yes' if jb_p > 0.05 else 'No',
    })

pd.DataFrame(diag_rows).set_index('Response')

---
## 6. Live Nowcast — Next COT Report

Using the filtered state at the last observed date $t_T$, we produce a 1-step-ahead forecast for $t_{T+1}$ with prediction intervals.

In [ ]:
# 1-step-ahead nowcast from the last filtered state
T_mat = mod['transition']                          # (k_states, k_states)
Z_mat = mod['design']                              # (k, k_states) or (k, k_states, T)
if Z_mat.ndim == 3:
    Z_now = Z_mat[:, :, -1]                        # use last time-step design
else:
    Z_now = Z_mat

alpha_T  = res.filtered_state[:, -1]               # filtered state at T (scaled)
P_T      = res.filtered_state_cov[:, :, -1]        # state covariance at T

# Predicted state at T+1
alpha_fc = T_mat @ alpha_T                         # (k_states,)
P_fc     = T_mat @ P_T @ T_mat.T + mod['state_cov']   # P_{T+1|T}

# Predicted observation at T+1
if BEST_MODEL == 'D: LL+Feat':
    # Need to supply X_{T+1} — use the latest available features (approximate)
    # In practice, these would be updated before the COT release
    x_fc  = X[-1]   # latest features (scaled)
    beta  = res.params[2*k:].reshape(k, n_features)
    y_fc_scaled = Z_now[:k, :k] @ alpha_fc[:k] + beta @ x_fc
else:
    y_fc_scaled = Z_now[:k, :k] @ alpha_fc[:k]

y_fc = y_fc_scaled * Y_std + Y_mean   # back-transform

# Predictive variance: Var[y_{T+1}] = Z P_{T+1|T} Z' + H
H_mat = mod['obs_cov']
pred_cov_fc = Z_now[:k, :k] @ P_fc[:k, :k] @ Z_now[:k, :k].T + H_mat
pred_std_fc = np.sqrt(np.diag(pred_cov_fc)) * Y_std

nowcast_date = pd.Timestamp(dates[-1]) + pd.Timedelta(weeks=1)
print(f'Nowcast for: {nowcast_date.date()}')
print(f'(Based on latest COT date: {pd.Timestamp(dates[-1]).date()})')

In [ ]:
# Nowcast table
nowcast_df = pd.DataFrame({
    'Response':   RESP_LABELS,
    'Nowcast':    y_fc.round(0).astype(int),
    'Std Error':  pred_std_fc.round(0).astype(int),
    '95% CI Low':  (y_fc - Z_95 * pred_std_fc).round(0).astype(int),
    '95% CI High': (y_fc + Z_95 * pred_std_fc).round(0).astype(int),
    'Last Actual': Y_raw[-1].round(0).astype(int),
    'Direction':  ['↑' if y_fc[i] > 0 else '↓' if y_fc[i] < 0 else '→'
                   for i in range(k)],
}).set_index('Response')

print(f'\n=== NOWCAST for {nowcast_date.date()} ===')
nowcast_df

In [ ]:
# Nowcast visualisation — last 52 weeks + forecast bar
N_PLOT = 52
fig, axes = plt.subplots(k, 1, figsize=(14, 3.5 * k), sharex=False)

for i, (ax, label) in enumerate(zip(axes, RESP_LABELS)):
    d_hist  = dates[-N_PLOT:]
    obs_h   = Y_raw[-N_PLOT:, i]
    fl_h    = filtered_orig[-N_PLOT:, i]
    pi_h    = pred_std_orig[-N_PLOT:, i]

    ax.bar(d_hist, obs_h, color='lightsteelblue', width=5, alpha=0.6, label='Observed Δ')
    ax.plot(d_hist, fl_h, color='darkorange', linewidth=1.2, label='Filtered')
    ax.fill_between(d_hist, fl_h - Z_95*pi_h, fl_h + Z_95*pi_h,
                    alpha=0.2, color='darkorange')

    # Nowcast point + 95% CI
    ax.bar([nowcast_date], [y_fc[i]], color='crimson', width=5, alpha=0.8, label='Nowcast')
    ax.errorbar([nowcast_date], [y_fc[i]],
                yerr=[[Z_95 * pred_std_fc[i]], [Z_95 * pred_std_fc[i]]],
                fmt='none', color='crimson', capsize=5, linewidth=2)

    ax.axhline(0, color='grey', linestyle=':', linewidth=0.8)
    ax.axvline(dates[-1], color='grey', linestyle='--', linewidth=1, alpha=0.7)
    ax.set_title(f'{label} — Last Year + Nowcast ({nowcast_date.date()})', fontweight='bold')
    ax.set_ylabel('Contracts')
    ax.legend(loc='upper right', fontsize=8)

plt.suptitle(f'{BEST_MODEL} — Live Nowcast with 95% Prediction Interval', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

---
## 7. Feature Contribution (if applicable)

In [ ]:
if BEST_MODEL == 'D: LL+Feat':
    beta_hat = res.params[2*k:].reshape(k, n_features)
    feat_short = [
        f.replace('prior_report_', 'pr_').replace('prior_', 'pr_')
        .replace('F1_RolledPrice_', 'F1_').replace('_change', 'Δ')
        .replace('_rolling_20D_volatility', '_Vol20D')
        .replace('cumulative_5D_', '5D_')
        .replace('SyntheticF1MinusF2_RolledPrice', 'Synth')
        for f in FEATURES
    ]
    # Attribution at the nowcast point
    x_last_orig = X_raw[-1]
    x_last_scaled = X[-1]
    contrib_scaled = beta_hat * x_last_scaled[np.newaxis, :]   # (k, p)

    fig, axes = plt.subplots(1, k, figsize=(16, 5))
    for i, (ax, label) in enumerate(zip(axes, RESP_LABELS)):
        vals  = contrib_scaled[i]
        cols  = ['green' if v > 0 else 'tomato' for v in vals]
        ax.barh(feat_short, vals, color=cols, alpha=0.8)
        ax.axvline(0, color='black', linewidth=0.8)
        ax.set_title(f'{label}\nFeature Contributions (nowcast)')
        ax.set_xlabel('Contribution (scaled units)')
    plt.suptitle(f'{BEST_MODEL} — Feature Attribution at Nowcast Point', fontsize=12, y=1.01)
    plt.tight_layout()
    plt.show()

elif BEST_MODEL == 'E: Dyn-Reg':
    print('Model E: time-varying β_t — see notebook 03 for full coefficient history.')
    print('Latest smoothed coefficients (β_T):')
    beta_T = res.smoothed_state[:, -1].reshape(k, n_features)
    beta_df = pd.DataFrame(beta_T, index=RESP_LABELS, columns=FEATURES)
    print(beta_df.round(4))

else:
    print(f'{BEST_MODEL} has no exogenous features.')

---
## 8. Export Artifacts

In [ ]:
# 1. Smoother estimates + uncertainty (full history)
smoother_df = pd.DataFrame({
    'tradeDate': pd.to_datetime(dates),
    **{f'smoothed_{r}': smoothed_orig[:, i] for i, r in enumerate(['net', 'long', 'short'])},
    **{f'smoothed_{r}_std': smoothed_std_orig[:, i] for i, r in enumerate(['net', 'long', 'short'])},
    **{f'filtered_{r}': filtered_orig[:, i] for i, r in enumerate(['net', 'long', 'short'])},
    **{f'pred_std_{r}': pred_std_orig[:, i] for i, r in enumerate(['net', 'long', 'short'])},
    **{f'actual_{r}': Y_raw[:, i] for i, r in enumerate(['net', 'long', 'short'])},
})
smoother_df.to_csv(OUT_DIR / 'kf_final_smoother_estimates.csv', index=False)
print(f'Smoother estimates saved: {smoother_df.shape}')

In [ ]:
# 2. Nowcast JSON
nowcast_dict = {
    'model':          BEST_MODEL,
    'nowcast_date':   str(nowcast_date.date()),
    'last_obs_date':  str(pd.Timestamp(dates[-1]).date()),
    'generated_at':   str(datetime.datetime.utcnow().isoformat()),
    'responses': {
        r: {
            'nowcast':      float(round(y_fc[i], 2)),
            'std_error':    float(round(pred_std_fc[i], 2)),
            'ci_95_low':    float(round(y_fc[i] - Z_95 * pred_std_fc[i], 2)),
            'ci_95_high':   float(round(y_fc[i] + Z_95 * pred_std_fc[i], 2)),
            'last_actual':  float(round(Y_raw[-1, i], 2)),
            'direction':    'up' if y_fc[i] > 0 else ('down' if y_fc[i] < 0 else 'flat'),
        }
        for i, r in enumerate(['net', 'long', 'short'])
    },
}
with open(OUT_DIR / 'kf_nowcast.json', 'w') as f:
    json.dump(nowcast_dict, f, indent=2)
print(json.dumps(nowcast_dict, indent=2))

In [ ]:
# 3. Model parameters JSON
params_dict = {
    'model':      BEST_MODEL,
    'n_obs':      int(len(Y)),
    'llf':        float(res.llf),
    'aic':        float(res.aic),
    'bic':        float(res.bic),
    'param_names': list(res.model.param_names),
    'params':      res.params.tolist(),
    'Y_mean':      Y_mean.tolist(),
    'Y_std':       Y_std.tolist(),
    'X_mean':      X_mean.tolist(),
    'X_std':       X_std.tolist(),
    'responses':   RESPONSES,
    'features':    FEATURES,
}
with open(OUT_DIR / 'kf_final_params.json', 'w') as f:
    json.dump(params_dict, f, indent=2)
print('Model parameters saved.')

---
## 9. Summary Dashboard

Final one-page summary of the winning KF model.

In [ ]:
# Load OOS metrics from notebook 04 scorecard
scorecard = pd.read_csv(OUT_DIR / 'kf_model_comparison_scorecard.csv', index_col=0)

fig = plt.figure(figsize=(16, 12))
gs  = fig.add_gridspec(3, 3, hspace=0.45, wspace=0.35)

# ---- Row 0: full-history smoother for each response ----
for i, label in enumerate(RESP_LABELS):
    ax = fig.add_subplot(gs[0, i])
    ax.bar(dates[-104:], Y_raw[-104:, i], color='lightsteelblue',
           width=5, alpha=0.5, label='Observed')
    ax.plot(dates[-104:], smoothed_orig[-104:, i], color='navy',
            linewidth=1.3, linestyle='--', label='Smoother')
    ax.fill_between(
        dates[-104:],
        smoothed_orig[-104:, i] - Z_95 * smoothed_std_orig[-104:, i],
        smoothed_orig[-104:, i] + Z_95 * smoothed_std_orig[-104:, i],
        alpha=0.15, color='navy'
    )
    ax.bar([nowcast_date], [y_fc[i]], color='crimson', width=5, alpha=0.9)
    ax.errorbar([nowcast_date], [y_fc[i]],
                yerr=[[Z_95 * pred_std_fc[i]], [Z_95 * pred_std_fc[i]]],
                fmt='none', color='crimson', capsize=4)
    ax.axhline(0, color='grey', linestyle=':', linewidth=0.7)
    ax.set_title(f'{label}\nSmoother + Nowcast', fontsize=9, fontweight='bold')
    ax.tick_params(labelsize=7)

# ---- Row 1: scorecard table ----
ax_tab = fig.add_subplot(gs[1, :])
ax_tab.axis('off')
cell_text  = [[f'{v:.4f}' if isinstance(v, float) else str(v)
               for v in row] for row in scorecard.values]
col_labels = scorecard.columns.tolist()
row_labels = scorecard.index.tolist()
tbl = ax_tab.table(
    cellText=cell_text,
    rowLabels=row_labels,
    colLabels=col_labels,
    loc='center',
    cellLoc='center',
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(8)
tbl.scale(1, 1.4)
ax_tab.set_title('Model Scorecard (OOS Walk-Forward)', fontsize=10, fontweight='bold', pad=12)

# ---- Row 2: nowcast bar ----
ax_now = fig.add_subplot(gs[2, :])
colors_nc = ['steelblue' if y_fc[i] > 0 else 'tomato' for i in range(k)]
ax_now.bar(RESP_LABELS, y_fc, color=colors_nc, alpha=0.8)
for i in range(k):
    ax_now.errorbar([RESP_LABELS[i]], [y_fc[i]],
                    yerr=[[Z_95*pred_std_fc[i]], [Z_95*pred_std_fc[i]]],
                    fmt='none', color='black', capsize=6, linewidth=2)
    ax_now.text(i, y_fc[i] + np.sign(y_fc[i]) * Z_95 * pred_std_fc[i] * 1.05,
                f'{y_fc[i]:+.0f}', ha='center', va='bottom' if y_fc[i] > 0 else 'top',
                fontsize=10, fontweight='bold')
ax_now.axhline(0, color='black', linewidth=0.8)
ax_now.set_title(f'Nowcast — {nowcast_date.date()}  (±95% PI)', fontsize=10, fontweight='bold')
ax_now.set_ylabel('Contracts (Δ)')

fig.suptitle(f'WTI MM COT Nowcasting — KF Final Model: {BEST_MODEL}', fontsize=14, fontweight='bold')
plt.savefig(OUT_DIR / 'kf_final_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print('Summary dashboard saved.')